# Lesson 1: Searching for BGC-Argo float profiles based on our research interests 🔍

### 🎯 Learning Objectives
In this lesson, we will use the Argo index file to search for the profiles we want based on our research interest. Specifically, we will filter the Argo index file by considering several factors:

* BGC parameters (e.g., we want profiles that include both nitrate and PAR)
* geographical range (e.g., we want profiles within the northwest Pacific)
* temporal range (e.g., we want profiles collected during 2024)
* sampling duration (e.g., we want floats that have collected profiles at least for two years)
* sampling frequency (e.g., we want floats that have collected profiles at least once per month)
* drift speed (e.g., we want floats that have drifted very slowly)

### 🛠️ Prerequisites
Before starting this lesson, you should have a brief understanding of what the BGC-Argo is all about. For this, we recommend reading [Bittig et al. (2019)](https://www.frontiersin.org/article/10.3389/fmars.2019.00502/full). In addition, we recommend reading about the structure and format of BGC-Argo data, which are nicely summarized on this [Data FAQ](https://www.go-bgc.org/data/data-faq) page provided by GO-BGC (Global Ocean Biogeochemistry Array).

### ❓ How to Use This Notebook
* 📚 **Read** the tutorial text blocks carefully, as they provide the essential background information behind the code.
* ▶️ **Run** each code cell sequentially by clicking the cell and pressing `Shift + Enter`.
* 📝 **Exercise** your knowledge! At the end of this notebook, we provide active learning exercises where you will need to write or modify the code yourself.

### Ready? Let's Get Started!
---

## 📚 Tutorial Part 1: Search by BGC parameters, longitudes, latitudes, and dates

### Import libraries

▶️ **Run** the cell below to import relevant Python libraries.

In [1]:
import urllib.request
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import cartopy.crs as ccrs
import cartopy.feature as cft 
from datetime import datetime, timedelta

### The index file

The original BGC-Argo data are hosted by the Global Data Assembly Center (GDAC) which has two nodes, namely [CORIORIS](https://data-argo.ifremer.fr) and [US-GODAE](https://usgodae.org/pub/outgoing/argo). To search and identify BGC-Argo float profiles of our interest, we will use the Argo index file for synthetic profiles (called [Sprof files](https://www.go-bgc.org/data/data-faq)). Click either [***this***](https://data-argo.ifremer.fr/argo_synthetic-profile_index.txt) or [***this***](https://usgodae.org/pub/outgoing/argo/argo_synthetic-profile_index.txt) to see the contents of the latest index file.

The index file provides the list of all available synthetic profiles collected by all kinds of Argo floats (i.e., Core, BGC, and Deep). It gives us the basic information for each profile, including when (`date`) and where (`latitude` and `longitude`) the profile was collected and which variables (`parameters`) were measured. We will use this index file to search for BGC-Argo floats and profiles we want.

▶️ **Run** the cell below to load the index file. 
* `skiprows` is used to ignore the first 8 rows of the original index file (which are just comments).
* `usecols` is used to load only the essential variables (i.e., file path, date, latitude, longitude, and parameters).
* We will try loading the index file from the CORIORIS node. If the node is down for some reason (e.g., maintenance), we will use the US-GODAE node.
* **NOTE** that the loading can take up to a minute or so becase the index file is large (e.g., 57 MB as of March 17, 2026).

In [2]:
try:
    # 1. Try the French server with a strict 10-second timeout
    print("Connecting to the French GDAC...")
    url1 = urllib.request.urlopen('https://data-argo.ifremer.fr/argo_synthetic-profile_index.txt', timeout=10)
    
    df_index = pd.read_csv(url1, 
                           skiprows=8, 
                           usecols=['file', 'date', 'latitude', 'longitude', 'parameters'])
    print("✅ Success!")

except Exception:
    # 2. If it stalls or fails, immediately fall back to the US server
    print("⚠️ French server timed out. Trying the US GDAC...")
    url2 = urllib.request.urlopen('https://usgodae.org/pub/outgoing/argo/argo_synthetic-profile_index.txt', timeout=10)
    
    df_index = pd.read_csv(url2, 
                           skiprows=8, 
                           usecols=['file', 'date', 'latitude', 'longitude', 'parameters'])
    print("✅ Success!")

df_index

Connecting to the French GDAC...
✅ Success!


,file,date,latitude,longitude,parameters
0,aoml/1900722/profiles/SD1900722_001.nc,2.006102e+13,-40.316,73.389,PRES TEMP PSAL DOXY
1,aoml/1900722/profiles/SD1900722_002.nc,2.006110e+13,-40.390,73.528,PRES TEMP PSAL DOXY
2,aoml/1900722/profiles/SD1900722_003.nc,2.006111e+13,-40.455,73.335,PRES TEMP PSAL DOXY
3,aoml/1900722/profiles/SD1900722_004.nc,2.006112e+13,-40.134,73.080,PRES TEMP PSAL DOXY
4,aoml/1900722/profiles/SD1900722_005.nc,2.006120e+13,-39.641,73.158,PRES TEMP PSAL DOXY
...,...,...,...,...,...
385157,meds/4902691/profiles/SR4902691_027.nc,2.026020e+13,47.627,-127.570,PRES TEMP PSAL DOXY DOWN_IRRADIANCE380 DOWN_IR...
385158,meds/4902691/profiles/SR4902691_028.nc,2.026021e+13,47.334,-128.050,PRES TEMP PSAL DOXY DOWN_IRRADIANCE380 DOWN_IR...
385159,meds/4902691/profiles/SR4902691_029.nc,2.026022e+13,47.378,-128.611,PRES TEMP PSAL DOXY DOWN_IRRADIANCE380 DOWN_IR...
385160,meds/4902691/profiles/SR4902691_031.nc,2.026032e+13,47.776,-128.319,PRES TEMP PSAL DOXY DOWN_IRRADIANCE380 DOWN_IR...


We see a spreadsheet showing the first and last five entries. Each row represents a profile collected by one of the Argo floats (Core, BGC, or Deep). The total number of rows represents the total number of profiles collected by the entire Argo program (enormous, isn't it?). As of March 19, 2026, there are 384,050 profiles.

### Filter by BGC parameters

The row `parameters` tells us the physical and biogeochemcal parameters included in the profiles. 

▶️ **Run** the cell below to get the full list of available parameters.

In [3]:
print(df_index['parameters'].str.split().explode().unique())

['PRES' 'TEMP' 'PSAL' 'DOXY' 'NITRATE' 'CHLA' 'CHLA_FLUORESCENCE' 'BBP700'
 'CDOM' 'PH_IN_SITU_TOTAL' 'DOWN_IRRADIANCE380' 'DOWN_IRRADIANCE443'
 'DOWN_IRRADIANCE490' 'DOWN_IRRADIANCE555' 'DOWNWELLING_PAR'
 'DOWN_IRRADIANCE412' 'DOXY2' 'DOXY_2' 'DOXY3' 'BBP532' 'DOXY_3'
 'TURBIDITY' 'CP660' 'BISULFIDE' 'DOWN_IRRADIANCE665' 'DOWN_IRRADIANCE670'
 'BBP700_2' 'BBP470' 'UP_RADIANCE412' 'UP_RADIANCE443' 'UP_RADIANCE490'
 'UP_RADIANCE555']


`PRES`, `TEMP`, and `PSAL` are the pressure (proxy for depth), temperature, and salinity, respectively.

As described in [Bittig et al. (2019)](https://www.frontiersin.org/article/10.3389/fmars.2019.00502/full), the following are the standard six BGC parameters:

* **CHLA**: chlorophyll-a (mg/m3)
* **BBP700**: back scattering coefficient at 700 nm (m-1)
* **NITRATE**: nitrate (umol/kg)
* **DOWNWELLING_PAR**: photosynthetically active radiation (W/m2)
* **DOXY**: dissolved oxygen (umol/kg)
* **PH_IN_SITU_TOTAL**: pH (-)

When a float is equipped with sensors that can measure all these six parameters, it is referred to as a **full-sensor** float.

As we see above, there are other BGC parameters such as `CDOM`.

▶️ **Run** the cell below to *define* the BGC parameters we want. Here, we will choose dissolved oxygen and back scattering coefficient at 700 nm.

In [4]:
bgc_parameters = ["DOXY", "BBP700"]

▶️ **Run** the cell below to *filter* the index file by BGC parameters

In [5]:
# Start with the full dataset
df_index_param = df_index.copy()

# Loop through each parameter
for param in bgc_parameters:
    # 3. Filter the dataframe and OVERWRITE it with the smaller, filtered version
    df_index_param = df_index_param[df_index_param['parameters'].str.contains(param)]

# Display the output
df_index_param

,file,date,latitude,longitude,parameters
1759,aoml/1901614/profiles/SD1901614_001.nc,2.025113e+13,15.018,-152.431,PRES TEMP PSAL DOXY CHLA CHLA_FLUORESCENCE BBP...
1760,aoml/1901614/profiles/SD1901614_002.nc,2.025121e+13,15.106,-153.135,PRES TEMP PSAL DOXY CHLA CHLA_FLUORESCENCE BBP...
1761,aoml/1901614/profiles/SD1901614_003.nc,2.025122e+13,15.261,-153.668,PRES TEMP PSAL DOXY CHLA CHLA_FLUORESCENCE BBP...
1762,aoml/1901614/profiles/SD1901614_004.nc,2.025123e+13,15.373,-154.037,PRES TEMP PSAL DOXY BBP700 CHLA CDOM PH_IN_SIT...
1763,aoml/1901614/profiles/SD1901614_005.nc,2.026011e+13,15.374,-154.581,PRES TEMP PSAL DOXY BBP700 CHLA CDOM PH_IN_SIT...
...,...,...,...,...,...
385135,meds/4902691/profiles/SR4902691_005.nc,2.025063e+13,49.036,-130.462,PRES TEMP PSAL DOXY DOWN_IRRADIANCE380 DOWN_IR...
385136,meds/4902691/profiles/SR4902691_006.nc,2.025071e+13,49.073,-130.398,PRES TEMP PSAL DOXY DOWN_IRRADIANCE380 DOWN_IR...
385137,meds/4902691/profiles/SR4902691_007.nc,2.025072e+13,49.037,-130.425,PRES TEMP PSAL DOXY DOWN_IRRADIANCE380 DOWN_IR...
385138,meds/4902691/profiles/SR4902691_008.nc,2.025073e+13,49.076,-130.410,PRES TEMP PSAL DOXY DOWN_IRRADIANCE380 DOWN_IR...


☝️ Notice that the number of rows (profiles) has decreased after the filtering.

### Filter by longitudes, latitudes, and dates

Next, we will apply the filter based on our spatial and temporal interests.

▶️ Run the cell below to *define* the spatial domain and temporal coverage we want:

* **lon0**: the easternmost longitude of the spatial domain (float: -180 to 180; **HOWEVER**, use 0 to 360 if the zonal range crosses the International Date Line).
* **lon1**: the westernmost longitude of the spatial domain (float: -180 to 180; **HOWEVER**, use 0 to 360 if the zonal range crosses the International Date Line. Make sure that **lon1** > **lon0**).
* **lat0**: the southernnmost latitude of the spatial domain (float: -90 to 90).
* **lat1**: the northernmost latitude of the spatial domain (float: -90 to 90. Make sure that **lat1** > **lat0**).
* **date0**: the first date of the temporal coverage (string: 'YYYY-MM-DD').
* **date1**: the last date of the temporal coverage (string: 'YYYY-MM-DD'. Make sure that **date1** > **date0**).

Here, we will limit to the northwest Pacific and for two years (2023-2024).

In [6]:
lon0 = 117
lon1 = 150
lat0 = 17
lat1 = 50
date0 = '2023-01-01'
date1 = '2024-12-31'

▶️ **Run** the cell below to filter the index file by longitudes and latitudes.

In [7]:
# Check
if lon0 > lon1:
    raise ValueError(f"lon0 cannot be greater than lon1.")

# Filter by longitudes
if lon0 > 180 or lon1 > 180:
    if lon0 < 0 or lon1 < 0:
        raise ValueError(
            f"lon0 and lon1 cannot be a mixture of <0 and >180." 
            f" You entered: {lon0} and {lon1}."
            f" Use either the -180 to 180 system or the 0 to 360 system."
        )
    else:
        # for the 0 to 360 system
        df_index_lon = df_index_param[(df_index_param['longitude'] % 360 >= lon0) & (df_index_param['longitude'] % 360 <= lon1)] 
else:
    # for the -180 to 180 system
    df_index_lon = df_index_param[(df_index_param['longitude'] >= lon0) & (df_index_param['longitude'] <= lon1)] 

if lat0 > lat1:
    raise ValueError(f"lat0 cannot be greater than lat1.")

# Filter by latitudes
df_index_lat = df_index_lon[(df_index_lon['latitude'] >= lat0) & (df_index_lon['latitude'] <= lat1)]

# Display the result
df_index_lat

,file,date,latitude,longitude,parameters
12626,aoml/2903887/profiles/SR2903887_086.nc,2.026013e+13,17.219,147.707,PRES TEMP PSAL DOXY PH_IN_SITU_TOTAL CHLA CHLA...
12627,aoml/2903887/profiles/SR2903887_087.nc,2.026021e+13,17.784,147.511,PRES TEMP PSAL DOXY PH_IN_SITU_TOTAL CHLA CHLA...
12628,aoml/2903887/profiles/SR2903887_088.nc,2.026022e+13,18.146,147.802,PRES TEMP PSAL DOXY PH_IN_SITU_TOTAL CHLA CHLA...
12629,aoml/2903887/profiles/SR2903887_092.nc,2.026023e+13,17.528,148.438,PRES TEMP PSAL DOXY PH_IN_SITU_TOTAL CHLA CHLA...
25929,aoml/4903850/profiles/SD4903850_001.nc,2.025072e+13,38.772,148.838,PRES TEMP PSAL DOXY CHLA CHLA_FLUORESCENCE BBP...
...,...,...,...,...,...
367340,jma/5906597/profiles/SR5906597_138.nc,2.025053e+13,43.158,147.880,PRES TEMP PSAL DOXY CHLA BBP700 CDOM NITRATE
367341,jma/5906597/profiles/SR5906597_139.nc,2.025053e+13,43.055,147.335,PRES TEMP PSAL DOXY CHLA BBP700 CDOM NITRATE
367342,jma/5906597/profiles/SR5906597_140.nc,2.025060e+13,43.129,147.317,PRES TEMP PSAL DOXY CHLA BBP700 CDOM NITRATE
367343,jma/5906597/profiles/SR5906597_141.nc,2.025061e+13,43.008,146.624,PRES TEMP PSAL DOXY CHLA BBP700 CDOM NITRATE


☝️ Notice that the number of rows (profiles) has decreased after the filtering.

▶️ **Run** the cell below to filter the index file by dates.

In [8]:
# Check
if date0 > date1:
    raise ValueError(f"date0 cannot be greater than date1.")

# For some reason, there are profiles with missing dates. Remove them first.
df_index_date = df_index_lat[df_index_lat['date'] >= 0]

# Convert date from float64 to string
df_index_date['date_str'] = df_index_date['date'].astype('int64').astype(str)

# Convert the string to a proper datetime object, which is easy to work with in Python.
df_index_date['datetime'] = pd.to_datetime(df_index_date['date_str'], format='%Y%m%d%H%M%S')

# Filter the index based on the temporal interest
df_index_date = df_index_date[(df_index_date['datetime'] >= f'{date0} 00:00:00') & (df_index_date['datetime'] <= f'{date1} 23:59:59')]

# Delete the columns `date` and `date_str` which are no longer needed
del df_index_date['date']
del df_index_date['date_str']

# Print
df_index_date

,file,latitude,longitude,parameters,datetime
123082,aoml/5906510/profiles/SD5906510_025.nc,28.861,136.833,PRES TEMP PSAL DOXY CHLA CHLA_FLUORESCENCE BBP...,2023-01-09 07:13:14
123083,aoml/5906510/profiles/SD5906510_026.nc,28.654,136.570,PRES TEMP PSAL DOXY CHLA CHLA_FLUORESCENCE BBP...,2023-01-19 13:05:06
123084,aoml/5906510/profiles/SD5906510_027.nc,28.408,136.452,PRES TEMP PSAL DOXY CHLA CHLA_FLUORESCENCE BBP...,2023-01-29 19:12:10
123085,aoml/5906510/profiles/SD5906510_028.nc,28.257,136.236,PRES TEMP PSAL DOXY CHLA CHLA_FLUORESCENCE BBP...,2023-02-09 01:04:18
123086,aoml/5906510/profiles/SD5906510_029.nc,27.966,135.831,PRES TEMP PSAL DOXY CHLA CHLA_FLUORESCENCE BBP...,2023-02-19 08:40:03
...,...,...,...,...,...
367153,jma/5906596/profiles/SR5906596_102.nc,48.866,149.866,PRES TEMP PSAL DOXY CHLA BBP700 CDOM NITRATE,2024-11-28 10:46:21
367154,jma/5906596/profiles/SR5906596_103.nc,48.867,149.854,PRES TEMP PSAL DOXY CHLA BBP700 CDOM NITRATE,2024-12-03 10:56:35
367155,jma/5906596/profiles/SR5906596_104.nc,48.876,149.831,PRES TEMP PSAL DOXY CHLA BBP700 CDOM NITRATE,2024-12-08 10:54:04
367158,jma/5906596/profiles/SR5906596_107.nc,48.796,149.954,PRES TEMP PSAL DOXY CHLA BBP700 CDOM NITRATE,2024-12-23 09:30:48


☝️ Notice that the number of rows (profiles) has decreased after the filtering.

### WMO: The unique identifier for Argo floats

All Argo floats are given unique seven-digit integers called the WMO (World Meteorological Organization) IDs, so that we can identify them easily. In the original Argo index file, this ID (which we will refer to as **wmo** for simplicity) is provided as the name for the second parental directory for the column `file`. This makes it a bit difficult to access wmo, so we will extract this information and create a new column called `wmo`.

▶️ **Run** the cell below to add `wmo` as an additional column.

In [9]:
# Make a copy of the filtered index file
df_index_wmo = df_index_date.copy()
# Split the file string by slashes and take the second item corresponding to wmo
df_index_wmo['wmo'] = df_index_wmo['file'].str.split('/').str[1]
# Display the output
df_index_wmo

,file,latitude,longitude,parameters,datetime,wmo
123082,aoml/5906510/profiles/SD5906510_025.nc,28.861,136.833,PRES TEMP PSAL DOXY CHLA CHLA_FLUORESCENCE BBP...,2023-01-09 07:13:14,5906510
123083,aoml/5906510/profiles/SD5906510_026.nc,28.654,136.570,PRES TEMP PSAL DOXY CHLA CHLA_FLUORESCENCE BBP...,2023-01-19 13:05:06,5906510
123084,aoml/5906510/profiles/SD5906510_027.nc,28.408,136.452,PRES TEMP PSAL DOXY CHLA CHLA_FLUORESCENCE BBP...,2023-01-29 19:12:10,5906510
123085,aoml/5906510/profiles/SD5906510_028.nc,28.257,136.236,PRES TEMP PSAL DOXY CHLA CHLA_FLUORESCENCE BBP...,2023-02-09 01:04:18,5906510
123086,aoml/5906510/profiles/SD5906510_029.nc,27.966,135.831,PRES TEMP PSAL DOXY CHLA CHLA_FLUORESCENCE BBP...,2023-02-19 08:40:03,5906510
...,...,...,...,...,...,...
367153,jma/5906596/profiles/SR5906596_102.nc,48.866,149.866,PRES TEMP PSAL DOXY CHLA BBP700 CDOM NITRATE,2024-11-28 10:46:21,5906596
367154,jma/5906596/profiles/SR5906596_103.nc,48.867,149.854,PRES TEMP PSAL DOXY CHLA BBP700 CDOM NITRATE,2024-12-03 10:56:35,5906596
367155,jma/5906596/profiles/SR5906596_104.nc,48.876,149.831,PRES TEMP PSAL DOXY CHLA BBP700 CDOM NITRATE,2024-12-08 10:54:04,5906596
367158,jma/5906596/profiles/SR5906596_107.nc,48.796,149.954,PRES TEMP PSAL DOXY CHLA BBP700 CDOM NITRATE,2024-12-23 09:30:48,5906596


☝️ There we have the new parameter (column) `wmo`!

### Functions

So far, we learned how to: 1) load the index file, 2) filter it by BGC parameters, longitudes, latitudes, and dates; and 3) add wmo as an additional parameter. If we want to repeat these steps with different inputs, it would be convenient to create and **functions**.

▶️ **Run** the cell below to define (`def`) a function that loads the original Argo index file from one of the GDAC nodes.

In [10]:
def load_index():
    try:
        # 1. Try the French server with a strict 10-second timeout
        print("Connecting to the French GDAC...")
        url1 = urllib.request.urlopen('https://data-argo.ifremer.fr/argo_synthetic-profile_index.txt', timeout=10)
        
        df_index = pd.read_csv(url1, 
                               skiprows=8, 
                               usecols=['file', 'date', 'latitude', 'longitude', 'parameters'])
        print("✅ Success!")
    
    except Exception:
        # 2. If it stalls or fails, immediately fall back to the US server
        print("⚠️ French server timed out. Trying the US GDAC...")
        url2 = urllib.request.urlopen('https://usgodae.org/pub/outgoing/argo/argo_synthetic-profile_index.txt', timeout=10)
        
        df_index = pd.read_csv(url2, 
                               skiprows=8, 
                               usecols=['file', 'date', 'latitude', 'longitude', 'parameters'])
        print("✅ Success!")
    
    return df_index

▶️ **Run** the cell below to load the index file using this function.

In [11]:
df_index = load_index()
df_index

Connecting to the French GDAC...
✅ Success!


,file,date,latitude,longitude,parameters
0,aoml/1900722/profiles/SD1900722_001.nc,2.006102e+13,-40.316,73.389,PRES TEMP PSAL DOXY
1,aoml/1900722/profiles/SD1900722_002.nc,2.006110e+13,-40.390,73.528,PRES TEMP PSAL DOXY
2,aoml/1900722/profiles/SD1900722_003.nc,2.006111e+13,-40.455,73.335,PRES TEMP PSAL DOXY
3,aoml/1900722/profiles/SD1900722_004.nc,2.006112e+13,-40.134,73.080,PRES TEMP PSAL DOXY
4,aoml/1900722/profiles/SD1900722_005.nc,2.006120e+13,-39.641,73.158,PRES TEMP PSAL DOXY
...,...,...,...,...,...
385157,meds/4902691/profiles/SR4902691_027.nc,2.026020e+13,47.627,-127.570,PRES TEMP PSAL DOXY DOWN_IRRADIANCE380 DOWN_IR...
385158,meds/4902691/profiles/SR4902691_028.nc,2.026021e+13,47.334,-128.050,PRES TEMP PSAL DOXY DOWN_IRRADIANCE380 DOWN_IR...
385159,meds/4902691/profiles/SR4902691_029.nc,2.026022e+13,47.378,-128.611,PRES TEMP PSAL DOXY DOWN_IRRADIANCE380 DOWN_IR...
385160,meds/4902691/profiles/SR4902691_031.nc,2.026032e+13,47.776,-128.319,PRES TEMP PSAL DOXY DOWN_IRRADIANCE380 DOWN_IR...


☝️ We see that we get the same result as we loaded the index at the beginning of this tutorial.

Similarly, let's define a function that filters the index file based on user inputs all at once.

▶️ **Run** the cell below.

In [12]:
def filter_index(df_index, lon0, lon1, lat0, lat1, date0, date1, bgc_parameters):
    """
    This function filters the index file by BGC parameters, longitudes, latitudes, and dates all at once.
    
    INPUT:
    * df_index: the original index file, which can be obtained by `load_index()`
    * lon0, lon1, lat0, lat1: the east, west, south, and north edges of the spatial domain
    * date0, date1: the beginning and end of the temporal coverage (string: 'YYYY-MM-DD')
    * bgc_parameters: the list of BGC parameters (string or list)

    OUTPUT:
    * df_index_out: the filtered index file
    """
    
    # Filter by parameters
    df_index_out = df_index.copy()
    for param in bgc_parameters:
        df_index_out = df_index_out[df_index_out['parameters'].str.contains(param)]
    
    # Filter by longitudes    
    if lon0 > lon1:
        raise ValueError(f"lon0 cannot be greater than lon1.")
    else:
        if lon0 > 180 or lon1 > 180:
            if lon0 < 0 or lon1 < 0:
                raise ValueError(
                    f"lon0 and lon1 cannot be a mixture of <0 and >180." 
                    f" You entered: {lon0} and {lon1}."
                    f" Use either the -180 to 180 system or the 0 to 360 system."
                )
            else:
                # for the 0 to 360 system
                df_index_out = df_index_out[(df_index_out['longitude'] % 360 >= lon0) & (df_index_out['longitude'] % 360 <= lon1)] 
        else:
            # for the -180 to 180 system
            df_index_out = df_index_out[(df_index_out['longitude'] >= lon0) & (df_index_out['longitude'] <= lon1)] 
    
    # Filter by latitudes
    if lat0 > lat1:
        raise ValueError(f"lat0 cannot be greater than lat1.")
    else:
        # Filter by latitudes
        df_index_out = df_index_out[(df_index_out['latitude'] >= lat0) & (df_index_out['latitude'] <= lat1)]

    # Check dates
    if date0 > date1:
        raise ValueError(f"date0 cannot be greater than date1.")
    
    # For some reason, there are profiles with missing dates. Remove them first.
    df_index_out = df_index_out[df_index_out['date'] >= 0]
    
    # Convert date from float64 to string
    df_index_out['date_str'] = df_index_out['date'].astype('int64').astype(str)
    
    # Convert the string to a proper datetime object
    df_index_out['datetime'] = pd.to_datetime(df_index_out['date_str'], format='%Y%m%d%H%M%S')
    
    # Filter by dates
    df_index_out = df_index_out[(df_index_out['datetime'] >= f'{date0} 00:00:00') & (df_index_out['datetime'] <= f'{date1} 23:59:59')]
    
    # Delete the columns `date` and `date_str` which are no longer needed
    del df_index_out['date']
    del df_index_out['date_str']

    # Add WMO
    df_index_out['wmo'] = df_index_out['file'].str.split('/').str[1]
    
    return df_index_out

☝️ As defined, this function requires the following inputs (arguments):

* **df_index**: the original Argo index file (which can be loaded with `load_index()`.
* **lon0, lon1, lat0, lat1, date0, date1, bgc_parameters**, all of which we defined earlier.

▶️ **Run** the cell below to filter the index based on our previously-defined inputs and return the filtered index. 

In [13]:
df_index_filtered = filter_index(df_index, lon0, lon1, lat0, lat1, date0, date1, bgc_parameters)
df_index_filtered

,file,latitude,longitude,parameters,datetime,wmo
123082,aoml/5906510/profiles/SD5906510_025.nc,28.861,136.833,PRES TEMP PSAL DOXY CHLA CHLA_FLUORESCENCE BBP...,2023-01-09 07:13:14,5906510
123083,aoml/5906510/profiles/SD5906510_026.nc,28.654,136.570,PRES TEMP PSAL DOXY CHLA CHLA_FLUORESCENCE BBP...,2023-01-19 13:05:06,5906510
123084,aoml/5906510/profiles/SD5906510_027.nc,28.408,136.452,PRES TEMP PSAL DOXY CHLA CHLA_FLUORESCENCE BBP...,2023-01-29 19:12:10,5906510
123085,aoml/5906510/profiles/SD5906510_028.nc,28.257,136.236,PRES TEMP PSAL DOXY CHLA CHLA_FLUORESCENCE BBP...,2023-02-09 01:04:18,5906510
123086,aoml/5906510/profiles/SD5906510_029.nc,27.966,135.831,PRES TEMP PSAL DOXY CHLA CHLA_FLUORESCENCE BBP...,2023-02-19 08:40:03,5906510
...,...,...,...,...,...,...
367153,jma/5906596/profiles/SR5906596_102.nc,48.866,149.866,PRES TEMP PSAL DOXY CHLA BBP700 CDOM NITRATE,2024-11-28 10:46:21,5906596
367154,jma/5906596/profiles/SR5906596_103.nc,48.867,149.854,PRES TEMP PSAL DOXY CHLA BBP700 CDOM NITRATE,2024-12-03 10:56:35,5906596
367155,jma/5906596/profiles/SR5906596_104.nc,48.876,149.831,PRES TEMP PSAL DOXY CHLA BBP700 CDOM NITRATE,2024-12-08 10:54:04,5906596
367158,jma/5906596/profiles/SR5906596_107.nc,48.796,149.954,PRES TEMP PSAL DOXY CHLA BBP700 CDOM NITRATE,2024-12-23 09:30:48,5906596


☝️ Notice that the result is identical as earlier.

In the future, if you forget what the function does, you can use `help()` to see the function description.

▶️ **Run** the cell below to see the description for `filter_index`.

In [14]:
help(filter_index)

Help on function filter_index in module __main__:

filter_index(df_index, lon0, lon1, lat0, lat1, date0, date1, bgc_parameters)
    This function filters the index file by BGC parameters, longitudes, latitudes, and dates all at once.

    INPUT:
    * df_index: the original index file, which can be obtained by `load_index()`
    * lon0, lon1, lat0, lat1: the east, west, south, and north edges of the spatial domain
    * date0, date1: the beginning and end of the temporal coverage (string: 'YYYY-MM-DD')
    * bgc_parameters: the list of BGC parameters (string or list)

    OUTPUT:
    * df_index_out: the filtered index file



## 📚 Tutorial Part 2: Search by sampling duration, frequency, and drift speed

Sometimes, we are looking for floats that have collected profiles for specific duration or at specific frequency. Furthermore, we may be interested in floats that are relatively stationary, which are useful for time-series studies and one-dimensional ocean modelling. 

We will consider these factors to further filter the index. Specifically, we define the following three additional inputs:

* **mindura**: the minimum duration of the sampling (in days)
* **minfreq**: the minimum frequency of the profiling cycle (in days)
* **maxdrif**: the maximum drift speed (in m/s). For learning more about Argo floats' drift, see [Katsumata and Yoshinari (2010)](https://doi.org/10.1007/s10872-010-0046-4).

▶️ **Run** the cell below to define these three criteria.

In [15]:
mindura = 100
minfreq = 14
maxdrif = 0.05

☝️ In this example, we want floats that have collected profiles for at least 100 days once every two weeks and drifting at a rate no more than 0.05 m/s.

▶️ **Run** the cell below to define a function that calculates the mean drift of a float based on longitudes, latitudes, and dates (technially, dates and times).

In [16]:
def calculate_float_speed(lon, lat, date):
    """
    This function calculates a rough estimate for the float's drift speed at parking depth, which is typically at 1,000 m.
    NOTE that this represents the overall mean speed and not the mean of the instantaneous speed (between cycles). 
    This prevents from division by zero when there are multiple sampling (especially in the first day of deployment).

    INPUT:
    * lon: longitudes of the profiles (array)
    * lat: latitiudes of the profiles (array)
    * date: dates of the profiles (array)

    OUTPUT:
    * speed: the roughly estimated drift speed of the float (m/s)
    """
    
    # Earth's radius in meters
    R = 6371000  
    # convert to radians
    lon_rad = np.radians(lon)#np.array(lon))
    lat_rad = np.radians(lat)#np.array(lat))
    # take the difference
    delta_lon = lon_rad.diff()
    delta_lat = lat_rad.diff()
    # Haversine formula
    a = np.sin(delta_lat / 2)**2 + np.cos(lat_rad[:-1]) * np.cos(lat_rad[1:]) * np.sin(delta_lon / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    distances = R * c  # in meters
    # Speed in meters per second
    speed = distances.sum() / date.diff().sum().total_seconds()
    # average speed
    return speed

▶️ **Run** the cell below to define a function that filters the index by the three criteria and save the filtered index file as a CSV file. Note that the three criteria are optional (if we do not care about these criteria, we do not need to provide them as input arguments). Also note that `note` is also an optional argument, which is used to create a distinguishable file name for the filtered index file.

In [17]:
def filter2_and_save_index(df_index, mindura = None, minfreq = None, maxdrif = None, note = 'default'):
    """
    This function filters the index file by the duration and frequency of the sampling as well as the drift speed and saves the filtered index as a CSV file.

    INPUT:
    * df_index: the index file (filtered by BGC parameters, longitudes, latitudes, and dates using `filter_index`) (format: pandas dataframe)
    * mindura: (format: integer)
    * minfreq: (format: integer)
    * maxdrif: (format: float)
    * note: this will be added to the file name (format: string)

    OUTPUT:
    * df_filtered2: (format: pandas dataframe). This will also be saved as a CSV file.
    """
    
    # Create a copy to create a further filtered index
    df_filtered2 = df_index.copy()
    
    for wmo, group in df_filtered2.groupby('wmo'):
        
        # Sort the group by time just to be safe
        group = group.sort_values('datetime')
    
        # Keep the profile if there is only one profile and mindura is set to 1 or None
        if len(group) == 1 and mindura in (None, 0, 1):
            valid_dura = True
            valid_freq = True
            valid_drif = True
        
        else: 
            # Filter out the float with a single profile
            if len(group) == 1:
                df_filtered2 = df_filtered2[df_filtered2['wmo'] != wmo]
            # Check the three criteria for multiple profiles
            else:  
                if mindura is None:
                    valid_dura = True
                else:
                    valid_dura = (group['datetime'].max() - group['datetime'].min()) > pd.to_timedelta(mindura, unit='D')
                if minfreq is None:
                    valid_freq = True
                else:
                    # Check the minimum frequency
                    valid_freq = group['datetime'].diff().max() < pd.to_timedelta(minfreq, unit='D')
                if maxdrif is None:
                    valid_drif = True
                else:
                    valid_drif = calculate_float_speed(group['longitude'], group['latitude'], group['datetime']) < maxdrif
    
        # Filter out the float's profiles if not true that all three are true
        if not (valid_dura and valid_freq and valid_drif):
            df_filtered2 = df_filtered2[df_filtered2['wmo'] != wmo]
    
    # The index=False prevents pandas from writing the 0, 1, 2 row numbers into the file.
    filename = f"argo_synthetic-profile_index_{note}.csv"
    df_filtered2.to_csv(filename, sep=",", index=False)

    print(f'\n{filename} has been saved to your current working directory.\n')
    return df_filtered2

▶️ **Run** the cell below to use this function.

In [18]:
filter2_and_save_index(df_index_filtered, mindura, minfreq, maxdrif)


argo_synthetic-profile_index_default.csv has been saved to your current working directory.



,file,latitude,longitude,parameters,datetime,wmo
123491,aoml/5906513/profiles/SD5906513_025.nc,27.139,139.571,PRES TEMP PSAL DOXY CHLA CHLA_FLUORESCENCE BBP...,2023-01-04 17:23:25,5906513
123492,aoml/5906513/profiles/SD5906513_026.nc,27.193,139.748,PRES TEMP PSAL DOXY CHLA CHLA_FLUORESCENCE BBP...,2023-01-15 02:00:45,5906513
123493,aoml/5906513/profiles/SD5906513_027.nc,27.255,139.714,PRES TEMP PSAL DOXY CHLA CHLA_FLUORESCENCE BBP...,2023-01-25 06:42:36,5906513
123494,aoml/5906513/profiles/SD5906513_028.nc,27.081,140.029,PRES TEMP PSAL DOXY CHLA CHLA_FLUORESCENCE BBP...,2023-02-04 11:05:10,5906513
123495,aoml/5906513/profiles/SD5906513_029.nc,26.829,139.993,PRES TEMP PSAL DOXY CHLA CHLA_FLUORESCENCE BBP...,2023-02-14 15:13:24,5906513
...,...,...,...,...,...,...
302989,csio/2902882/profiles/SR2902882_174.nc,20.532,139.938,PRES TEMP PSAL DOXY CHLA BBP532 BBP700 DOWN_IR...,2024-12-11 02:17:22,2902882
302990,csio/2902882/profiles/SR2902882_175.nc,20.622,139.968,PRES TEMP PSAL DOXY CHLA BBP532 BBP700 DOWN_IR...,2024-12-16 02:11:21,2902882
302991,csio/2902882/profiles/SR2902882_176.nc,20.718,139.975,PRES TEMP PSAL DOXY CHLA BBP532 BBP700 DOWN_IR...,2024-12-21 02:20:16,2902882
302992,csio/2902882/profiles/SR2902882_177.nc,20.786,139.957,PRES TEMP PSAL DOXY CHLA BBP532 BBP700 DOWN_IR...,2024-12-26 02:08:23,2902882


☝️ Notice that the number of profiles has decreased after this filtering. Also check your directory to make sure that this filtered index has been saved as a CSV file. This is the final filtered index file and we will use it as a basis for our next lessons.

**This is the end of the tutorials for Lesson 1. Hope you enjoyed it!**

---

## 📝 Exercises

Now that we have learned how to use the index file to search for floats and profiles based on our research interests, let's do a few exercises by considering different use cases. Keep in mind that we can use the three functions we have developed for convenience (`load_index`, `filter_index`, and `filter2_and_save_index`).

### Exercise 1: Create and save a filtered index file that provides a list of profiles that measured chlorophyll-a on March 1, 2026 in the global ocean.

**NOTE**: When saving the index file using `filter2_and_save_index`, use `note = 'exercise1'` provided below as the argument for `note`, so that it will not overwrite the CSV file created above.

In [19]:
lon0 = -180
lon1 = 180
lat0 = -90
lat1 = 90
date0 = '2026-03-01'
date1 = '2026-03-01'
bgc_parameters = 'CHLA'
mindura = None
minfreq = None
maxdrif = None
note = 'ex1'
df_index = load_index()
df_filter = filter_index(df_index, lon0, lon1, lat0, lat1, date0, date1, bgc_parameters)
filter2_and_save_index(df_filter, mindura, minfreq, maxdrif, note)

Connecting to the French GDAC...
✅ Success!

argo_synthetic-profile_index_ex1.csv has been saved to your current working directory.



,file,latitude,longitude,parameters,datetime,wmo
1768,aoml/1901614/profiles/SD1901614_010.nc,14.033,-156.924,PRES TEMP PSAL DOXY BBP700 CHLA CDOM PH_IN_SIT...,2026-03-01 18:29:37,1901614
2852,aoml/1902380/profiles/SR1902380_151.nc,17.604,-47.276,PRES TEMP PSAL DOXY CHLA CHLA_FLUORESCENCE BBP...,2026-03-01 06:23:51,1902380
5358,aoml/1902647/profiles/SR1902647_046.nc,-48.920,-134.482,PRES TEMP PSAL DOXY CHLA CHLA_FLUORESCENCE BBP...,2026-03-01 00:31:55,1902647
6039,aoml/1902746/profiles/SR1902746_003.nc,-50.384,107.228,PRES TEMP PSAL DOXY CHLA CHLA_FLUORESCENCE BBP...,2026-03-01 11:58:16,1902746
8715,aoml/2903440/profiles/SR2903440_050.nc,-37.681,-98.117,PRES TEMP PSAL DOXY CHLA CHLA_FLUORESCENCE BBP...,2026-03-01 17:59:45,2903440
...,...,...,...,...,...,...
378895,meds/4902601/profiles/SR4902601_090.nc,30.765,-75.554,PRES PSAL TEMP DOXY CHLA BBP700,2026-03-01 00:38:00,4902601
379772,meds/4902622/profiles/SR4902622_070.nc,38.138,-63.379,PRES PSAL TEMP DOXY CHLA BBP700,2026-03-01 02:34:00,4902622
383577,meds/4902681/profiles/SR4902681_298.nc,49.625,-143.922,PRES PSAL TEMP DOXY CHLA BBP700 NITRATE,2026-03-01 02:52:00,4902681
384945,meds/4902687/profiles/SR4902687_145.nc,59.018,-46.964,PRES TEMP PSAL DOXY DOWN_IRRADIANCE380 DOWN_IR...,2026-03-01 21:06:00,4902687


### Exercise 2: Create and save a filtered index file that only contains profiles collected by the *full-sensor* BGC-Argo floats in the Indian Ocean at least once per month and at least for a year between 2018 and 2025.

In [20]:
lon0 = 25
lon1 = 115
lat0 = -40
lat1 = 10
date0 = '2018-01-01'
date1 = '2025-12-31'
bgc_parameters = ['DOXY', 'NITRATE', 'CHLA', 'BBP700', 'PH_IN_SITU_TOTAL', 'DOWNWELLING_PAR']
mindura = 365
minfreq = 30
maxdrif = None
note = 'ex2'
df_index = load_index()
df_filter = filter_index(df_index, lon0, lon1, lat0, lat1, date0, date1, bgc_parameters)
filter2_and_save_index(df_filter, mindura, minfreq, maxdrif, note)

Connecting to the French GDAC...
✅ Success!

argo_synthetic-profile_index_ex2.csv has been saved to your current working directory.



,file,latitude,longitude,parameters,datetime,wmo
327334,csiro/5905531/profiles/SD5905531_001.nc,-22.267,112.419,PRES TEMP PSAL DOXY DOWN_IRRADIANCE380 DOWN_IR...,2022-11-27 13:32:00,5905531
327335,csiro/5905531/profiles/SD5905531_002.nc,-22.251,112.386,PRES TEMP PSAL DOXY DOWN_IRRADIANCE380 DOWN_IR...,2022-11-29 01:44:00,5905531
327336,csiro/5905531/profiles/SD5905531_003.nc,-22.237,112.343,PRES TEMP PSAL DOXY DOWN_IRRADIANCE380 DOWN_IR...,2022-11-30 13:36:00,5905531
327337,csiro/5905531/profiles/SD5905531_004.nc,-22.210,112.303,PRES TEMP PSAL DOXY DOWN_IRRADIANCE380 DOWN_IR...,2022-12-02 01:37:00,5905531
327338,csiro/5905531/profiles/SD5905531_005.nc,-22.191,112.269,PRES TEMP PSAL DOXY DOWN_IRRADIANCE380 DOWN_IR...,2022-12-03 13:38:00,5905531
...,...,...,...,...,...,...
327407,csiro/5905531/profiles/SD5905531_074.nc,-28.640,106.280,PRES TEMP PSAL DOXY DOWN_IRRADIANCE380 DOWN_IR...,2024-07-17 23:43:00,5905531
327408,csiro/5905531/profiles/SD5905531_075.nc,-29.250,106.370,PRES TEMP PSAL DOXY DOWN_IRRADIANCE380 DOWN_IR...,2024-07-28 03:35:00,5905531
327409,csiro/5905531/profiles/SD5905531_076.nc,-29.459,106.562,PRES TEMP PSAL DOXY DOWN_IRRADIANCE380 DOWN_IR...,2024-08-07 07:29:00,5905531
327410,csiro/5905531/profiles/SD5905531_077.nc,-29.872,106.265,PRES TEMP PSAL DOXY DOWN_IRRADIANCE380 DOWN_IR...,2024-08-17 15:31:00,5905531


### Exercise 3: Create and save a filtered index file that only contains profiles that measured both dissolved oxygen and nitrate and drifted at a speed of nore more than 0.05 m/s in the North Pacific between 2019 and 2022.
**TIP**: Be careful how you define the longitude range, as it crosses the over the International Date Line.

In [21]:
lon0 = 120
lon1 = 260
lat0 = 0
lat1 = 60
date0 = '2019-01-01'
date1 = '2022-12-31'
bgc_parameters = ['DOXY', 'NITRATE']
mindura = None
minfreq = None
maxdrif = 0.05
note = 'ex3'
df_index = load_index()
df_filter = filter_index(df_index, lon0, lon1, lat0, lat1, date0, date1, bgc_parameters)
filter2_and_save_index(df_filter, mindura, minfreq, maxdrif, note)

Connecting to the French GDAC...
✅ Success!

argo_synthetic-profile_index_ex3.csv has been saved to your current working directory.



,file,latitude,longitude,parameters,datetime,wmo
21409,aoml/4903026/profiles/SD4903026_000.nc,36.753,-122.512,PRES TEMP PSAL DOXY PH_IN_SITU_TOTAL CHLA BBP7...,2022-02-05 00:07:28,4903026
21410,aoml/4903026/profiles/SD4903026_001.nc,36.762,-122.512,PRES TEMP PSAL DOXY PH_IN_SITU_TOTAL CHLA CHLA...,2022-02-05 03:37:24,4903026
21411,aoml/4903026/profiles/SD4903026_002.nc,36.769,-122.513,PRES TEMP PSAL DOXY PH_IN_SITU_TOTAL CHLA CHLA...,2022-02-05 07:06:41,4903026
21412,aoml/4903026/profiles/SD4903026_003.nc,36.783,-122.525,PRES TEMP PSAL DOXY PH_IN_SITU_TOTAL CHLA CHLA...,2022-02-05 10:44:49,4903026
21413,aoml/4903026/profiles/SD4903026_004.nc,36.798,-122.530,PRES TEMP PSAL DOXY PH_IN_SITU_TOTAL CHLA CHLA...,2022-02-05 14:18:35,4903026
...,...,...,...,...,...,...
300253,csio/2902822/profiles/SD2902822_196.nc,22.083,162.888,PRES TEMP PSAL DOXY CHLA BBP700 CDOM DOWN_IRRA...,2022-08-04 02:07:16,2902822
300254,csio/2902822/profiles/SD2902822_197.nc,22.227,162.793,PRES TEMP PSAL DOXY CHLA BBP700 CDOM DOWN_IRRA...,2022-08-07 02:11:23,2902822
300255,csio/2902822/profiles/SD2902822_198.nc,22.417,162.689,PRES TEMP PSAL DOXY CHLA BBP700 CDOM DOWN_IRRA...,2022-08-10 02:11:21,2902822
300256,csio/2902822/profiles/SD2902822_199.nc,22.605,162.566,PRES TEMP PSAL DOXY CHLA BBP700 CDOM DOWN_IRRA...,2022-08-13 02:00:23,2902822



---

This is the end of the lesson. If you are using **Binder**, don't forget to dowload the files created in this lesson before you lose connection!

Well done 🎉 Take a break 💤, have another cup ☕, and move to the next lesson ✍️ when you are ready 💪

While your memory is fresh, please feel free to provide your user experience on this lesson by visiting [this link](https://forms.gle/oAGmz5RTW4Pp46bt7). Thanks!